# GT Keypoint Reconstruction

This notebook loads all key frames and their ground truth poses, then reconstructs all key points using the GT pose. Key points are sampled in each frame and stored in the first frame coordinate system. This notebook uses GT poses to transform them back to each key frame's coordinate system.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from typing import Optional, List, Tuple, Dict

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

# Import project modules
from point2pose.io.sources.dataset.datareader import Ho3dReader, YcbineoatReader
from point2pose.utils.transform import transform_pts, inverse_SE3

# Use interactive backend for rotatable 3D plots
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('matplotlib', 'widget')
except:
    try:
        if ipython is not None:
            ipython.run_line_magic('matplotlib', 'notebook')
    except:
        import matplotlib
        matplotlib.use('inline')
        print("Note: Interactive 3D plots not available. Using static plots.")

In [ ]:
def find_meta_data_path(
    register_folder: str = None,
    results_dir: Optional[str] = None,
    video_name: Optional[str] = None,
) -> Optional[str]:
    """Find meta_data.npz file in expected locations (similar to plot_registration_stats.py)."""
    script_dir = os.path.dirname(os.path.abspath('.'))
    project_root = os.path.abspath('..')
    
    meta_data_paths = []
    
    # Check new results folder structure first
    if results_dir and video_name:
        new_structure_path = os.path.join(results_dir, video_name, "meta_data", "meta_data.npz")
        meta_data_paths.append(new_structure_path)
        default_results_dir = os.path.join(project_root, "results", "ho3d_single")
        if results_dir != default_results_dir:
            meta_data_paths.append(os.path.join(default_results_dir, video_name, "meta_data", "meta_data.npz"))
    
    # Existing paths for backward compatibility
    if register_folder:
        meta_data_paths.extend([
            os.path.join(register_folder, "meta_data.npz"),
            os.path.join(os.path.dirname(register_folder), "meta_data.npz"),
        ])
    
    # Common debug locations
    meta_data_paths.extend([
        os.path.join(project_root, "meta_data", "meta_data.npz"),
        os.path.join(project_root, "debug", "meta_data", "meta_data.npz"),
        os.path.join(project_root, "debug", "pipeline", "meta_data", "meta_data.npz"),
        os.path.join(os.getcwd(), "meta_data", "meta_data.npz"),
        os.path.join(os.getcwd(), "debug", "pipeline", "meta_data", "meta_data.npz"),
    ])
    
    # Also check old typo for backward compatibility
    meata_data_paths = [
        os.path.join(register_folder, "meata_data.npz") if register_folder else None,
        os.path.join(project_root, "debug", "pipeline", "meta_data", "meata_data.npz"),
    ]
    meata_data_paths = [p for p in meata_data_paths if p is not None]
    
    all_paths = meta_data_paths + meata_data_paths
    for path in all_paths:
        if path and os.path.exists(path):
            print(f"Found meta_data.npz at: {path}")
            return path
    
    print(f"Warning: Could not find meta_data.npz in {len(all_paths)} locations")
    return None

def unpack_ragged(name: str, store: dict, dim: int = -1) -> list:
    """Unpack ragged array data from NPZ storage format.
    
    Args:
        name: Base name of the ragged field (e.g., "obj_key_points")
        store: Dictionary loaded from NPZ file
        dim: Dimension to reshape to (-1 for 1D, 2 for 2D points, 3 for 3D points)
    
    Returns:
        List of arrays, one per frame
    """
    data_key = f"{name}_data"
    offsets_key = f"{name}_offsets"
    lengths_key = f"{name}_lengths"
    
    if data_key not in store or offsets_key not in store or lengths_key not in store:
        return []
    
    data = store[data_key]
    offsets = store[offsets_key]
    lengths = store[lengths_key]
    
    out = []
    for off, L in zip(offsets, lengths):
        flat_data = data[off : off + L]
        
        # Handle empty or invalid data
        if L == 0 or len(flat_data) == 0:
            if dim == 3:
                out.append(np.array([]).reshape(0, 3))
            elif dim == 2:
                out.append(np.array([]).reshape(0, 2))
            else:
                out.append(np.array([]))
            continue
        
        # Reshape based on dimension
        if dim == 3:
            if len(flat_data) % 3 != 0:
                print(f"Warning: {name} data length {len(flat_data)} not divisible by 3")
                out.append(flat_data)
            else:
                out.append(flat_data.reshape(-1, 3))
        elif dim == 2:
            if len(flat_data) % 2 != 0:
                print(f"Warning: {name} data length {len(flat_data)} not divisible by 2")
                out.append(flat_data)
            else:
                out.append(flat_data.reshape(-1, 2))
        else:
            out.append(flat_data)
    
    return out

In [ ]:
# Configuration - Update these paths for your data
# Option 1: Specify results directory and video name (new structure)
results_dir = "/home/justin/code/point-to-pose/results/ho3d_single"  # e.g., "/path/to/results/ho3d_single"
video_name = "MPM12"   # e.g., "MPM10"

# Option 2: Specify register folder (old structure)
register_folder = None  # e.g., "/path/to/debug/pipeline/register"

# Option 3: Direct path to meta_data.npz
meta_data_path_override = None  # e.g., "/path/to/meta_data/meta_data.npz"

# Dataset paths (for loading GT poses)
ho3d_root = "/home/justin/data/HO3D_V3/evaluation/"  # e.g., "/mnt/9a72c439-d0a7-45e8-8d20-d7a235d02763/DATASET/HO3D"
video_dir = None  # e.g., "/mnt/9a72c439-d0a7-45e8-8d20-d7a235d02763/DATASET/HO3D/MPM10"

# If video_dir is not provided, try to infer from ho3d_root and video_name
if video_dir is None and ho3d_root and video_name:
    potential_video_dir = os.path.join(ho3d_root, video_name)
    if os.path.exists(potential_video_dir) and os.path.exists(os.path.join(potential_video_dir, "rgb")):
        video_dir = potential_video_dir
        print(f"Using video_dir: {video_dir}")

# Find meta_data.npz
if meta_data_path_override and os.path.exists(meta_data_path_override):
    meta_data_path = meta_data_path_override
else:
    meta_data_path = find_meta_data_path(
        register_folder=register_folder,
        results_dir=results_dir,
        video_name=video_name,
    )

if meta_data_path is None:
    raise FileNotFoundError("Could not find meta_data.npz. Please update the configuration above.")

print(f"Loading metadata from: {meta_data_path}")
data = np.load(meta_data_path, allow_pickle=True)
print(f"Loaded metadata with keys: {list(data.keys())[:20]}...")

In [ ]:
# Extract frame IDs and key point data
if "frame_id" not in data:
    raise KeyError("No frame_id field in meta_data.npz")

frame_ids = data["frame_id"]
print(f"Total frames in metadata: {len(frame_ids)}")
print(f"Frame ID range: {frame_ids.min()} to {frame_ids.max()}")

# Unpack key points and their frame IDs
obj_key_points_list = unpack_ragged("obj_key_points", data, dim=3)
obj_key_point_frames_list = unpack_ragged("obj_key_point_frames", data, dim=-1)

print(f"Unpacked {len(obj_key_points_list)} frames of key points")
if len(obj_key_points_list) > 0:
    print(f"First frame has {len(obj_key_points_list[0])} key points")

In [ ]:
# Extract all unique key frames (frames where new points were sampled)
# Key frames are identified by the unique frame IDs in obj_key_point_frames
all_key_frame_ids = set()

# Also check if is_key_frame field exists (alternative way to identify key frames)
if "is_key_frame" in data:
    is_key_frame = data["is_key_frame"]
    if isinstance(is_key_frame, np.ndarray) and len(is_key_frame) == len(frame_ids):
        # Handle boolean or integer array
        if is_key_frame.dtype == bool:
            key_frames_from_flag = frame_ids[is_key_frame].tolist()
        else:
            key_frames_from_flag = frame_ids[is_key_frame != 0].tolist()
        print(f"Found {len(key_frames_from_flag)} key frames from is_key_frame flag")
        # Merge with existing key frames
        key_frames = sorted(list(set(key_frames_from_flag + [0])))
        print(f"Total unique key frames after merging: {len(key_frames)}")
        print(f"key frames from flag: {key_frames[:20]}{'...' if len(key_frames) > 20 else ''}")

In [ ]:
# Create datareader for loading GT poses
reader = None
if video_dir and os.path.exists(video_dir):
    # Try to infer ho3d_root if not provided
    actual_ho3d_root = ho3d_root
    if ho3d_root is None or not os.path.exists(os.path.join(ho3d_root, "models")):
        # Try parent directory of video_dir
        potential_root = os.path.dirname(video_dir)
        if os.path.exists(os.path.join(potential_root, "models")):
            actual_ho3d_root = potential_root
        else:
            # Try common locations
            common_paths = [
                "/mnt/9a72c439-d0a7-45e8-8d20-d7a235d02763/DATASET/HO3D",
            ]
            for path in common_paths:
                if os.path.exists(path) and os.path.exists(os.path.join(path, "models")):
                    actual_ho3d_root = path
                    break
    
    if actual_ho3d_root and os.path.exists(actual_ho3d_root):
        try:
            print(f"Creating Ho3dReader with video_dir={video_dir}, ho3d_root={actual_ho3d_root}")
            reader = Ho3dReader(video_dir, actual_ho3d_root)
            print(f"Successfully created reader with {len(reader)} frames")
        except Exception as e:
            print(f"Warning: Could not create Ho3dReader: {e}")
            import traceback
            traceback.print_exc()
    else:
        print(f"Warning: ho3d_root not found or invalid: {actual_ho3d_root}")
else:
    print(f"Warning: video_dir not found: {video_dir}")

if reader is None:
    print("Warning: Could not create datareader. GT pose loading will be skipped.")

In [ ]:
# load reg data

reg_key_points_list = unpack_ragged("reg_key_points", data, dim=3)  # list of (Mi,3) float arrays
reg_cur3d_list = unpack_ragged("reg_curr3d", data, dim=3)            # list of (Mi,3) float arrays
reg_key_points_idx = unpack_ragged("reg_key_points_idx", data)  # list of (Mi,) int arrays
obj_key_points_list = unpack_ragged("obj_key_points", data, dim=3)  # list of (Mi,3) float arrays
obj_key_point_frames = unpack_ragged("obj_key_point_frames", data)  # list of (Mi,3) float arrays
uncertainties = unpack_ragged("uncertainties", data)  # list of (Mi,) float arrays. measured uncertainty at each frame

In [ ]:
# Load GT poses for all key frames
gt_poses = {}
gt_poses_valid = {}

if reader is not None:
    for kf_id in key_frames:
        if kf_id < len(reader):
            gt_pose = reader.get_gt_pose(kf_id)
            if gt_pose is not None:
                gt_poses[kf_id] = gt_pose
                gt_poses_valid[kf_id] = True
            else:
                gt_poses_valid[kf_id] = False
        else:
            gt_poses_valid[kf_id] = False
    
    print(f"Loaded GT poses for {len(gt_poses)} out of {len(key_frames)} key frames")
    if len(gt_poses) < len(key_frames):
        missing = [kf for kf in key_frames if kf not in gt_poses]
        print(f"Missing GT poses for key frames: {missing[:10]}{'...' if len(missing) > 10 else ''}")
else:
    print("Skipping GT pose loading (no datareader available)")

In [ ]:
# Find the reference frame (first frame with GT pose, or frame 0)
reference_frame_id = None
reference_gt_pose = None

if reader is not None:
    # Try frame 0 first
    if 0 < len(reader):
        ref_pose = reader.get_gt_pose(0)
        if ref_pose is not None:
            reference_frame_id = 0
            reference_gt_pose = ref_pose
        else:
            # Find first frame with GT pose
            for i in range(len(reader)):
                pose = reader.get_gt_pose(i)
                if pose is not None:
                    reference_frame_id = i
                    reference_gt_pose = pose
                    break
    
    if reference_gt_pose is not None:
        print(f"Using frame {reference_frame_id} as reference frame for coordinate system")
    else:
        print("Warning: No reference frame with GT pose found")

In [ ]:
# Reconstruct key points for each key frame using GT pose
# Loop through all key frames, extract initialized key points, transform to first frame

all_reconstructed_points = []  # List to store all transformed points
all_reconstructed_points_fid = []
has_gt_flags = []  # Flags indicating if each point has GT pose

est_pose_list = data["obj_pose"]  # Estimated poses

for kf_id in key_frames:
    # Collect all key points initialized in this key frame
    kp_points_kf = []
    
    mask = obj_key_point_frames[kf_id] == kf_id
    kp_points_kf = obj_key_points_list[kf_id][mask]
    # print(f"kp_points_kf: {kp_points_kf.shape}")
    
    kp_points_kf = np.vstack(kp_points_kf)  # (N, 3) in key frame coordinates
    # for kp in kp_points_kf:
    #     if np.isnan(kp).any():
    #         print(f"kp: {kp}")
    est_pose = est_pose_list[kf_id]
    # Transform to first frame
    if kf_id in gt_poses:
        # Use GT pose
        transform_kf_to_ref = reference_gt_pose @ inverse_SE3(gt_poses[kf_id]) @ est_pose
        has_gt = True
    else:
        # Use estimated pose
        frame_idx_in_data = np.where(frame_ids == kf_id)[0]
        if len(frame_idx_in_data) > 0:
            transform_kf_to_ref = np.eye(4)
            has_gt = False
        else:
            continue
    
    # Transform points
    kp_points_h = np.hstack([kp_points_kf, np.ones((len(kp_points_kf), 1))])
    kp_points_ref = (transform_kf_to_ref @ kp_points_h.T).T[:, :3]
    
    all_reconstructed_points.append(kp_points_ref)
    all_reconstructed_points_fid.extend([kf_id] * len(kp_points_ref))
    has_gt_flags.extend([has_gt] * len(kp_points_ref))

# Combine all points into one array
all_reconstructed_points = np.vstack(all_reconstructed_points) if len(all_reconstructed_points) > 0 else np.empty((0, 3))
all_reconstructed_points_no_nan = all_reconstructed_points[~np.isnan(all_reconstructed_points).any(axis=1)]
# Convert to numpy array before boolean indexing
all_reconstructed_points_fid = np.array(all_reconstructed_points_fid)
all_reconstructed_points_fid_no_nan = all_reconstructed_points_fid[~np.isnan(all_reconstructed_points).any(axis=1)]
has_gt_flags = np.array(has_gt_flags)


In [ ]:
# Helper function to create color map for frames (similar to plot_registration_stats.py)
def create_frame_color_map(frame_ids):
    """Create a color map for frame IDs using tab10 colormap."""
    unique_frames = sorted(list(set(frame_ids)))
    
    try:
        import matplotlib
        cmap_get = getattr(getattr(matplotlib, "colormaps", matplotlib.cm), "get_cmap")
        tab10_colors = cmap_get("tab10")
        
        frame_to_color = {}
        for i, fid in enumerate(unique_frames):
            color_idx = i if i < 3 else i + 1  # Skip red (index 3)
            frame_to_color[fid] = tab10_colors(color_idx % 10)[:3]
    except (AttributeError, ImportError, KeyError):
        # Fallback palette
        palette = [
            (0.2, 0.2, 1.0),  # blue
            (0.0, 0.7, 0.3),  # green
            (1.0, 0.6, 0.0),  # orange
            (0.6, 0.0, 0.8),  # purple
            (0.0, 0.7, 0.7),  # cyan
            (0.6, 0.6, 0.0),  # yellow
        ]
        frame_to_color = {
            fid: palette[i % len(palette)]
            for i, fid in enumerate(unique_frames)
        }
    
    return frame_to_color

# Summary statistics
print("\n=== Summary ===")
print(f"Total key frames: {len(key_frames)}")
print(f"Key frames with GT poses: {len(gt_poses)}")
print(f"Key frames with reconstructed points: {len(all_reconstructed_points)}")

if len(all_reconstructed_points) > 0:
    print(f"Total reconstructed points: {all_reconstructed_points.shape[0]}")


In [ ]:
# Visualize all reconstructed key points in the FIRST FRAME, colored by the frame they were added
if len(all_reconstructed_points_no_nan) > 0:
    # Create color map for frames
    frame_to_color = create_frame_color_map(all_reconstructed_points_fid_no_nan)
    unique_frames = sorted(list(set(all_reconstructed_points_fid_no_nan)))
    
    # Create figure - make it bigger
    fig = plt.figure(figsize=(15, 15))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot key points grouped by frame ID (colored by when they were added)
    for fid in unique_frames:
        mask = all_reconstructed_points_fid_no_nan == fid
        if np.any(mask):
            ax.scatter(
                all_reconstructed_points_no_nan[mask, 0],
                all_reconstructed_points_no_nan[mask, 1],
                all_reconstructed_points_no_nan[mask, 2],
                c=[frame_to_color[fid]],
                alpha=0.7,
                s=30,
                label=f"Frame {fid}",
                marker="o",
                edgecolors="black",
                linewidths=0.5,
            )
    
    # Plot origin
    ax.scatter([0], [0], [0], c='red', s=100, marker='x', linewidths=3, label='Origin')
    
    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    ax.set_zlabel('Z', fontsize=12)
    ax.set_title('All Reconstructed Key Points in First Frame (Colored by Frame Added)', fontsize=14, fontweight='bold')
    
    # Limit legend to first 20 frames for readability
    handles, labels = ax.get_legend_handles_labels()
    if len(handles) > 21:  # 20 frames + origin
        ax.legend(handles[:21], labels[:21], loc="upper left", fontsize=8, ncol=2)
    else:
        ax.legend(loc="upper left", fontsize=8, ncol=2)
    
    ax.grid(True, alpha=0.3)
    
    # Set equal aspect ratio
    max_range = np.array([
        all_reconstructed_points_no_nan[:, 0].max() - all_reconstructed_points_no_nan[:, 0].min(),
        all_reconstructed_points_no_nan[:, 1].max() - all_reconstructed_points_no_nan[:, 1].min(),
        all_reconstructed_points_no_nan[:, 2].max() - all_reconstructed_points_no_nan[:, 2].min()
    ]).max() / 2.0
    mid_x = (all_reconstructed_points_no_nan[:, 0].max() + all_reconstructed_points_no_nan[:, 0].min()) * 0.5
    mid_y = (all_reconstructed_points_no_nan[:, 1].max() + all_reconstructed_points_no_nan[:, 1].min()) * 0.5
    mid_z = (all_reconstructed_points_no_nan[:, 2].max() + all_reconstructed_points_no_nan[:, 2].min()) * 0.5
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)
    ax.set_box_aspect((1, 1, 1))

    plt.tight_layout()
    plt.show()
    
    print(f"Visualized {len(all_reconstructed_points)} key points from {len(unique_frames)} frames")
else:
    print("No key points to visualize in first frame")

In [ ]:
# Export reconstructed key points for further analysis
# This creates a dictionary that can be saved or used in other notebooks

# export_data = {
#     'key_frames': key_frames,
#     'gt_poses': gt_poses,
#     'reconstructed_key_points': reconstructed_key_points,  # Points sampled in each frame
#     'all_reconstructed_per_frame': all_reconstructed_per_frame,  # All points transformed to each frame
#     'all_key_points_first_frame': all_key_points_first_frame,  # All points in first frame coords
#     'all_key_point_frame_ids_first_frame': all_key_point_frame_ids_first_frame,  # Frame IDs for points
#     'key_points_per_frame': key_points_per_frame,  # Original indices and points grouped by frame
#     'reference_frame_id': reference_frame_id,
#     'reference_gt_pose': reference_gt_pose,
# }

# print("Export data structure created with keys:", list(export_data.keys()))
# print("\nYou can now use 'export_data' in subsequent cells for further analysis.")

In [ ]:
frame_id = 1544

gt_pose_first_frame = reader.get_gt_pose(0)
gt_pose_frame = reader.get_gt_pose(frame_id)
gt_pose_frame = gt_pose_first_frame @ inverse_SE3(gt_pose_frame)

reg_cur3d = reg_cur3d_list[frame_id]
key_points_idx = reg_key_points_idx[frame_id]
key_points = reg_key_points_list[frame_id]

gt_key_points = all_reconstructed_points
uncertainties_frame = uncertainties[frame_id]

# Transform reg_cur3d to first frame and compute errors
reg_cur3d_first = (gt_pose_frame @ np.hstack([reg_cur3d, np.ones((len(reg_cur3d), 1))]).T).T[:, :3]
corresponding_gt = gt_key_points[key_points_idx]
errors = np.linalg.norm(reg_cur3d_first - corresponding_gt, axis=1)

print(f"Mean error: {np.mean(errors):.4f}m, Median: {np.median(errors):.4f}m")

# Plot error of each point
plt.figure(figsize=(12, 6))
plt.scatter(range(len(errors)), errors, alpha=0.6, s=20)
plt.axhline(np.mean(errors), color='r', linestyle='--', label=f'Mean: {np.mean(errors):.4f}m')
plt.xlabel('Point Index')
plt.ylabel('Error (m)')
plt.title(f'Error per Point - Frame {frame_id}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Plot error vs uncertainty
    
uncertainty_masked = uncertainties_frame[key_points_idx]
plt.figure(figsize=(10, 6))
plt.scatter(uncertainty_masked, errors, alpha=0.6, s=20)
plt.xlabel('Uncertainty')
plt.ylabel('Error (m)')
plt.title(f'Error vs Uncertainty - Frame {frame_id}')
plt.grid(True, alpha=0.3)
plt.show()
    
# 3D visualization of observed points and GT key points
fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(reg_cur3d_first[:, 0], reg_cur3d_first[:, 1], reg_cur3d_first[:, 2], 
           c='blue', s=30, alpha=0.6, label='Observed Points')
ax.scatter(corresponding_gt[:, 0], corresponding_gt[:, 1], corresponding_gt[:, 2], 
           c='red', s=30, alpha=0.6, label='GT Key Points')
# Draw lines connecting corresponding points
for i in range(min(100, len(errors))):  # Show max 100 lines for clarity
    ax.plot([reg_cur3d_first[i, 0], corresponding_gt[i, 0]],
            [reg_cur3d_first[i, 1], corresponding_gt[i, 1]],
            [reg_cur3d_first[i, 2], corresponding_gt[i, 2]], 'g-', alpha=0.3, linewidth=1)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title(f'3D Visualization - Frame {frame_id}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# 3D visualization colored by uncertainty
fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(reg_cur3d_first[:, 0], reg_cur3d_first[:, 1], reg_cur3d_first[:, 2], 
                     c=uncertainty_masked, cmap='viridis', s=30, alpha=0.6, label='Observed Points (colored by uncertainty)')
ax.scatter(corresponding_gt[:, 0], corresponding_gt[:, 1], corresponding_gt[:, 2], 
           c='red', s=30, alpha=0.6, label='GT Key Points')
# Draw lines connecting corresponding points
for i in range(min(100, len(errors))):  # Show max 100 lines for clarity
    ax.plot([reg_cur3d_first[i, 0], corresponding_gt[i, 0]],
            [reg_cur3d_first[i, 1], corresponding_gt[i, 1]],
            [reg_cur3d_first[i, 2], corresponding_gt[i, 2]], 'g-', alpha=0.3, linewidth=1)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title(f'3D Visualization Colored by Uncertainty - Frame {frame_id}')
plt.colorbar(scatter, ax=ax, label='Uncertainty')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



In [ ]:
# Compute error vs uncertainty statistics across all frames
all_errors = []
all_uncertainties = []
frame_ids_processed = []

gt_pose_first_frame = reader.get_gt_pose(0)

# Process all frames that have data
for fid in range(len(reg_cur3d_list)):
    if fid >= len(uncertainties) or fid >= len(reg_key_points_idx):
        continue
    
    try:
        # Get GT pose for this frame
        gt_pose_frame = reader.get_gt_pose(fid)
        if gt_pose_frame is None:
            continue
        
        gt_pose_frame = gt_pose_first_frame @ inverse_SE3(gt_pose_frame)
        
        reg_cur3d = reg_cur3d_list[fid]
        key_points_idx = reg_key_points_idx[fid]
        uncertainties_frame = uncertainties[fid]
        
        if len(reg_cur3d) == 0 or len(key_points_idx) == 0:
            continue
        
        # Transform reg_cur3d to first frame and compute errors
        reg_cur3d_first = (gt_pose_frame @ np.hstack([reg_cur3d, np.ones((len(reg_cur3d), 1))]).T).T[:, :3]
        corresponding_gt = all_reconstructed_points[key_points_idx]
        errors = np.linalg.norm(reg_cur3d_first - corresponding_gt, axis=1)
        
        # Get uncertainties for the corresponding key points
        uncertainty_masked = uncertainties_frame[key_points_idx]
        
        # Filter out NaN values
        valid_mask = ~(np.isnan(errors) | np.isnan(uncertainty_masked))
        errors_valid = errors[valid_mask]
        uncertainty_valid = uncertainty_masked[valid_mask]
        
        if len(errors_valid) > 0:
            all_errors.extend(errors_valid)
            all_uncertainties.extend(uncertainty_valid)
            frame_ids_processed.extend([fid] * len(errors_valid))
    except Exception as e:
        continue

all_errors = np.array(all_errors)
all_uncertainties = np.array(all_uncertainties)

print(f"Processed {len(frame_ids_processed)} points from {len(set(frame_ids_processed))} frames")
print(f"Error range: [{np.min(all_errors):.4f}, {np.max(all_errors):.4f}] m")
print(f"Uncertainty range: [{np.min(all_uncertainties):.4f}, {np.max(all_uncertainties):.4f}]")
print(f"Correlation coefficient: {np.corrcoef(all_errors, all_uncertainties)[0, 1]:.4f}")

# Create comprehensive statistics plot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Scatter plot of all errors vs uncertainties
ax = axes[0, 0]
ax.scatter(all_uncertainties, all_errors, alpha=0.3, s=5)
ax.set_xlabel('Uncertainty')
ax.set_ylabel('Error (m)')
ax.set_title('Error vs Uncertainty (All Frames)')
ax.grid(True, alpha=0.3)

# 2. Binned statistics: mean error per uncertainty bin
ax = axes[0, 1]
n_bins = 20
uncertainty_bins = np.linspace(np.min(all_uncertainties), np.max(all_uncertainties), n_bins + 1)
bin_indices = np.digitize(all_uncertainties, uncertainty_bins)
bin_means = [np.mean(all_errors[bin_indices == i]) for i in range(1, len(uncertainty_bins))]
bin_stds = [np.std(all_errors[bin_indices == i]) for i in range(1, len(uncertainty_bins))]
bin_centers = (uncertainty_bins[:-1] + uncertainty_bins[1:]) / 2
ax.errorbar(bin_centers, bin_means, yerr=bin_stds, fmt='o-', capsize=3, capthick=1)
ax.set_xlabel('Uncertainty (bin center)')
ax.set_ylabel('Mean Error (m)')
ax.set_title('Mean Error per Uncertainty Bin')
ax.grid(True, alpha=0.3)

# 3. Histogram of errors colored by uncertainty quartiles
ax = axes[1, 0]
uncertainty_quartiles = np.percentile(all_uncertainties, [25, 50, 75])
colors = ['blue', 'green', 'orange', 'red']
labels = [f'Q1 (≤{uncertainty_quartiles[0]:.3f})', 
          f'Q2 ({uncertainty_quartiles[0]:.3f}-{uncertainty_quartiles[1]:.3f})',
          f'Q3 ({uncertainty_quartiles[1]:.3f}-{uncertainty_quartiles[2]:.3f})',
          f'Q4 (>{uncertainty_quartiles[2]:.3f})']
for i in range(4):
    if i == 0:
        mask = all_uncertainties <= uncertainty_quartiles[0]
    elif i == 1:
        mask = (all_uncertainties > uncertainty_quartiles[0]) & (all_uncertainties <= uncertainty_quartiles[1])
    elif i == 2:
        mask = (all_uncertainties > uncertainty_quartiles[1]) & (all_uncertainties <= uncertainty_quartiles[2])
    else:
        mask = all_uncertainties > uncertainty_quartiles[2]
    ax.hist(all_errors[mask], bins=30, alpha=0.5, color=colors[i], label=labels[i])
ax.set_xlabel('Error (m)')
ax.set_ylabel('Frequency')
ax.set_title('Error Distribution by Uncertainty Quartiles')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Box plot: error distribution per uncertainty quartile
ax = axes[1, 1]
box_data = []
box_labels = []
for i in range(4):
    if i == 0:
        mask = all_uncertainties <= uncertainty_quartiles[0]
    elif i == 1:
        mask = (all_uncertainties > uncertainty_quartiles[0]) & (all_uncertainties <= uncertainty_quartiles[1])
    elif i == 2:
        mask = (all_uncertainties > uncertainty_quartiles[1]) & (all_uncertainties <= uncertainty_quartiles[2])
    else:
        mask = all_uncertainties > uncertainty_quartiles[2]
    box_data.append(all_errors[mask])
    box_labels.append(f'Q{i+1}')
ax.boxplot(box_data, labels=box_labels)
ax.set_xlabel('Uncertainty Quartile')
ax.set_ylabel('Error (m)')
ax.set_title('Error Distribution by Uncertainty Quartile')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()